# LifeMate — Zalo AI Challenge 2026
Zero-shot baseline → SFT fine-tune → submission

**Model**: Qwen3-4B (4.02B params) | **GPU**: T4 | **Budget**: 30s/sample

## 1. Setup

In [ ]:
!pip install unsloth transformers accelerate peft bitsandbytes trl datasets librosa -q
!rm -rf /tmp/repo && git clone https://github.com/Justinbiahoi-coding/zalo-lifemate-2026.git /tmp/repo
!cp /tmp/repo/*.py /tmp/repo/*.json .
!ls *.py *.json

In [ ]:
import glob, os
# Tự dò ROOT thay vì hardcode: đường dẫn /kaggle/input chứa tên tài khoản,
# nên bản hardcode chết ngay khi đổi acc.
hits = glob.glob("/kaggle/input/**/train/train_tools_catalog.json", recursive=True)
assert hits, ("Chưa gắn dataset vào notebook. Add Input -> tìm dataset lifemate.\n"
              "Hiện /kaggle/input có: " + str(os.listdir("/kaggle/input")))
ROOT = os.path.dirname(os.path.dirname(hits[0]))
print("ROOT =", ROOT)
print(sorted(os.listdir(ROOT)))

## 2. Generate SFT Training Data

In [ ]:
!python make_sft.py --root {ROOT} --dev split_dev.json --out sft_train.jsonl
!python augment_sft.py --input sft_train.jsonl --output sft_aug.jsonl --copies 2

## 3. Fine-tune (QLoRA NF4)

In [ ]:
# 4038 mẫu x 2 epoch = 505 steps (~5.5h). 5 epoch = 1262 steps = VƯỢT giới hạn 12h Kaggle.
# Step count là ngân sách; data đa dạng là miễn phí -> augment nhiều, epoch ít.
!python train_sft.py --data sft_aug.jsonl --epochs 2 --r 64 --out lora_out

## 4. Submit — Public Test (634 mẫu, có ASR)

Chạy **trước** eval: tổng notebook ~10h/12h, biên mỏng. Nếu hết giờ thì mất
chẩn đoán (chạy lại rẻ, dùng lora_out đã lưu) chứ không mất bài nộp.

In [ ]:
import os
assert os.path.exists("lora_out/adapter_config.json"), "lora_out không có — training hỏng"
!python baseline.py --root {ROOT} --split public_test \
    --lora lora_out --no-think --asr --out submission.jsonl

In [ ]:
# ── 5. Eval chẩn đoán — dev set 347 mẫu, dùng gold audio_text (không ASR) ──
# So với các lần trước: gate pass là chỉ số chính, không phải điểm tổng.
!python baseline.py --root {ROOT} --split train --dev split_dev.json \
    --lora lora_out --no-think --out sub_ft.jsonl --score

### Hao hụt do ASR

Cùng dev set, nhưng phiên âm thật thay vì dùng `audio_text` vàng.
Khoảng cách với cell trên = **giá thật của ASR** — con số public_test thực sự chấm.

In [ ]:
!python baseline.py --root {ROOT} --split train --dev split_dev.json \
    --lora lora_out --no-think --asr --out sub_ft_asr.jsonl --score --limit 100